In [1]:

%pip install \
    "strands-agents>=1.60.0,<2.0.0" \
    "strands-agents-tools>=0.8.9,<0.9.0" \
    "langchain>=1.4.0,<2.0.0" \
    "langchain-aws>=1.8.0,<2.0.0" \
    "langgraph>=1.2.0,<2.0.0" \
    "boto3>=1.40.0"


INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.5/923.5 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 43.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.0 MB/s  0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-con

In [1]:

from strands import Agent
from strands.models.openai import OpenAIModel
import os
from google.colab import userdata

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

model = OpenAIModel(
    client_args={
        "api_key": userdata.get("GROQ_API_KEY"),
        "base_url": os.getenv(
            "GROQ_BASE_URL",
            "https://api.groq.com/openai/v1"
        ),
        "max_retries": 5,
    },
    model_id=MODEL_ID,
    params={"extra_body": {"include_reasoning": False}},
)

# agent = Agent(model=model, callback_handler=None )

# response = agent(
#     """
#       Explain groq in a bulleted list.
#     """
# )

# print(response)


**GROQ – “Graph‑Relational Object Queries” (Sanity.io’s query language)**  

- **What it is**  
  - A powerful, JSON‑centric query language created by Sanity for fetching and shaping content stored in a Sanity dataset.  
  - Designed to be expressive like GraphQL but with a simpler, more flexible syntax that works directly on the raw document structure.

- **Core concepts**  
  - **Documents** – top‑level objects in Sanity (e.g., `post`, `author`, `product`).  
  - **Fields** – properties inside a document; can be primitives, arrays, objects, or references to other documents.  
  - **References** – pointers (`_ref`) that link one document to another, enabling relational queries.  
  - **Projection** – the part of the query that selects which fields to return (similar to a “select” clause).  

- **Basic syntax**  
  - `*` – selects all documents.  
  - `[ _type == "post" ]` – filter clause (like a WHERE).  
  - `{ title, slug, "authorName": author->name }` – projection block; `->` deref

In [9]:
%pip install -U langchain langchain-groq langgraph langgraph-checkpoint-sqlite python-dotenv  langchain-openai

In [11]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

load_dotenv()

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")

llm = ChatOpenAI(
    model=MODEL_ID,
    api_key=userdata.get("GROQ_API_KEY"),
    base_url=os.getenv(
        "GROQ_BASE_URL",
        "https://api.groq.com/openai/v1"
    ),
    max_retries=5,
    model_kwargs={
        "extra_body": {
            "include_reasoning": False
        }
    }
)

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful math assistant.",
    checkpointer=checkpointer
)

In [18]:
if not userdata.get("TAVILY_API_KEY"):
    print("⚠️  TAVILY_API_KEY not set — the news and sentiment agents cannot search the web.")
else:
    print("✓ TAVILY_API_KEY is configured")

✓ TAVILY_API_KEY is configured


Setting Up Yahoo Finace


In [19]:
import yfinance as yf
from strands import Agent, ToolContext, tool
from strands_tools.tavily import tavily_search


@tool(context=True)
async def web_search(query: str, tool_context: ToolContext, time_range: str = "month") -> dict | str:
    """Search the web with Tavily and return the top 3 results.

    Args:
        query: What to search for
        time_range: How recent the results should be: day, week, month or year
    """
    searches = tool_context.agent.state.get("searches") or 0  # as_tool() resets state for each task

    if searches >= 2:
        return "Search limit reached: answer with the results you already have."
    tool_context.agent.state.set("searches", searches + 1)
    return await tavily_search(query=query, max_results=3, time_range=time_range)

**Building Custom Tools**

In [20]:
def _num(value, money=False, pct=False):
    """Format numbers safely — Yahoo sometimes leaves fields empty."""
    if not isinstance(value, (int, float)):
        return "N/A"
    if pct:
        return f"{value * 100:.2f}%"
    return f"${value:,.2f}" if money else f"{value:,.2f}"


@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price and key metrics using Yahoo Finance.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA', 'MSFT')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Stock: {symbol}\n"
            f"Current Price: {_num(info.get('currentPrice') or info.get('regularMarketPrice'), money=True)}\n"
            f"Market Cap: {_num(info.get('marketCap'), money=True)}\n"
            f"P/E Ratio: {_num(info.get('trailingPE'))}\n"
            f"52 Week High: {_num(info.get('fiftyTwoWeekHigh'), money=True)}\n"
            f"52 Week Low: {_num(info.get('fiftyTwoWeekLow'), money=True)}\n"
            f"Average Volume: {_num(info.get('averageVolume'))}\n"
            f"Sector: {info.get('sector', 'N/A')}\n"
            f"Industry: {info.get('industry', 'N/A')}"
        )
    except Exception as e:
        return f"Error fetching stock data for {symbol}: {e}"


@tool
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """Get historical stock price data.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
        period: One of 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    """
    try:
        hist = yf.Ticker(symbol).history(period=period)
        if hist.empty:
            return f"No historical data available for {symbol}"
        start_price = hist["Close"].iloc[0]
        end_price = hist["Close"].iloc[-1]
        change_pct = (end_price - start_price) / start_price * 100
        return (
            f"Historical Data for {symbol} ({period}):\n"
            f"Starting Price: ${start_price:.2f}\n"
            f"Ending Price: ${end_price:.2f}\n"
            f"Change: {change_pct:+.2f}%\n"
            f"Highest: ${hist['High'].max():.2f}\n"
            f"Lowest: ${hist['Low'].min():.2f}\n"
            f"Average Volume: {hist['Volume'].mean():,.0f}\n\n"
            f"Recent 5-day trend:\n{hist[['Close', 'Volume']].tail().to_string()}"
        )
    except Exception as e:
        return f"Error fetching historical data for {symbol}: {e}"

@tool
def get_company_financials(symbol: str) -> str:
    """Get company financial metrics: revenue, margins, growth and leverage.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Financial Metrics for {symbol}:\n"
            f"Revenue: {_num(info.get('totalRevenue'), money=True)}\n"
            f"Revenue Growth: {_num(info.get('revenueGrowth'), pct=True)}\n"
            f"Gross Profit: {_num(info.get('grossProfits'), money=True)}\n"
            f"Profit Margin: {_num(info.get('profitMargins'), pct=True)}\n"
            f"Operating Margin: {_num(info.get('operatingMargins'), pct=True)}\n"
            f"EBITDA: {_num(info.get('ebitda'), money=True)}\n"
            f"Earnings Growth: {_num(info.get('earningsGrowth'), pct=True)}\n"
            f"Return on Equity: {_num(info.get('returnOnEquity'), pct=True)}\n"
            f"Debt to Equity: {_num(info.get('debtToEquity'))}\n"
            f"Free Cash Flow: {_num(info.get('freeCashflow'), money=True)}"
        )
    except Exception as e:
        return f"Error fetching financial data for {symbol}: {e}"


print(get_stock_price(symbol="AAPL"))  # tools are still plain functions you can call

Stock: AAPL
Current Price: $348.02
Market Cap: $5,079,066,214,400.00
P/E Ratio: 39.91
52 Week High: $351.73
52 Week Low: $243.42
Average Volume: 42,874,346.00
Sector: Technology
Industry: Consumer Electronics


In [21]:
financial_tools = [get_stock_price, get_stock_history, get_company_financials, web_search]

Worker Agent


In [22]:
news_agent = Agent(
    name="news_agent",
    description="Researches recent news and developments about companies and industries.",
    model=model,
    system_prompt="""You are a news research specialist. Search for recent news and developments about companies and industries.

When searching:
- Focus on recent news (use time_range='week' or 'month')
- Prefer reliable sources such as major news outlets
- Summarize key developments and trends
- Include specific facts, dates, and figures when available
- Make at most two web_search calls: Groq's free tier caps each request at 8,000 tokens""",
    tools=[web_search],
    callback_handler=None,
)

financial_agent = Agent(
    name="financial_agent",
    description="Analyzes stock prices, price history, and company financial metrics.",
    model=model,
    system_prompt="""You are a financial analysis specialist. Analyze financial metrics, stock performance, and company valuations.

When analyzing:
- Use get_stock_price for current prices and key metrics
- Use get_stock_history for price trends
- Use get_company_financials for revenue, earnings, and growth
- Compare companies within the same sector and give specific numbers
- Look at no more than three companies, and use web_search at most once""",
    tools=financial_tools,
    callback_handler=None,
)
sentiment_agent = Agent(
    name="sentiment_agent",
    description="Analyzes market and investor sentiment from forums, social media, and commentary.",
    model=model,
    system_prompt="""You are a market sentiment analyst. Analyze market sentiment from social media, forums, and investor discussions.

When analyzing sentiment:
- Look for both bullish and bearish opinions
- Identify common concerns and excitement points
- Distinguish retail from institutional sentiment
- Summarize the overall market mood
- Make at most two web_search calls: Groq's free tier caps each request at 8,000 tokens""",
    tools=[web_search],
    callback_handler=None,
)
print("✓ Worker agents created")

✓ Worker agents created


**Supervisor Agent**

In [23]:
supervisor = Agent(
    name="research_supervisor",
    model=model,
    system_prompt="""You are a market research coordinator. When asked about investment opportunities:

1. Work out what information is needed
2. Delegate to the specialist agents:
   - news_agent: recent developments and industry news
   - financial_agent: financial metrics and performance data
   - sentiment_agent: market sentiment and investor opinions
3. Synthesize their responses into one balanced recommendation

This is educational analysis, not financial advice — say so briefly at the end.
Do not ask the user clarifying questions: pick sensible defaults (for example, the sector's leading companies) and delegate straight away.""",
    tools=[news_agent.as_tool(), financial_agent.as_tool(), sentiment_agent.as_tool()],
)
print("✓ Supervisor created")

✓ Supervisor created


In [24]:
question = "IS it Right Time to Invest in Apple Stock?"
print(f"Question: {question}\n")

response = supervisor(question)

Question: IS it Right Time to Invest in Apple Stock?


Tool #1: news_agent



Tool #2: financial_agent



Tool #3: sentiment_agent


**Is Now the Right Time to Invest in Apple (AAPL)? – A Balanced View**  

---

## 1. What the data tells us  

| Area | Key Findings (Oct 2026) | Why it matters |
|------|------------------------|----------------|
| **Recent news** | • iPhone 16 Pro “Titanium” launch – 12 % YoY shipment lift, premium pricing ($1,199‑$1,399).<br>• Services revenue up 9 % YoY to $21 bn (≈ 20 % of total revenue).<br>• Vision Pro 2.0 prototype leak – cheaper, slimmer, 8K dual‑eye displays; analysts see a potential $15‑$20 bn revenue tail‑wind by 2029.<br>• Supply‑chain: Foxconn +10 % capacity, TSMC +3 % 5‑nm/3‑nm output, dual‑foundry strategy reducing risk.<br>• Regulatory pressure: EU antitrust “statement of objections” on App Store fees; US FTC injunction on Apple Pay data sharing. | Shows strong product momentum (hardware & services) and solid supply‑chain, but regulatory risk is rising. |
| **Financial metrics (Q3 FY 2027)** | • Revenue $466.8 bn (+16.4 % YoY).<br>• Gross margin 48.7 %; net profit marg